# CFN Pipeline Minimal (Reproducible + Evidence)

This notebook runs a compact train->eval pipeline and now includes an optional DFDC-boost stage to improve cross-domain accuracy.


In [ ]:
# 0) Environment + reusable runner (run first)
from pathlib import Path
import os, sys, json, subprocess, random, shutil
from datetime import datetime

LOCAL_ROOT = Path('/Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend')
COLAB_ROOT = Path('/content/CausalX-Project/backend')
PROJECT_ROOT = LOCAL_ROOT if LOCAL_ROOT.exists() else COLAB_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

VENV_PY = PROJECT_ROOT / '.venv' / 'bin' / 'python'
PY_BIN = str(VENV_PY if VENV_PY.exists() else Path(sys.executable))

os.environ['PYTHONPATH'] = str(PROJECT_ROOT)
os.environ.setdefault('CFN_USE_EMBEDDINGS', 'true')
os.environ.setdefault('CFN_W2V2_MODEL', 'WAV2VEC2_BASE')
os.environ.setdefault('CFN_EMB_MODEL_PATH', str(PROJECT_ROOT / 'models' / 'cfn_emb.pth'))
os.environ.setdefault('CFN_VISUAL_TCN_PATH', str(PROJECT_ROOT / 'models' / 'visual_tcn.pth'))
os.environ.setdefault('MEDIAPIPE_DISABLE_GPU', '1')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = PROJECT_ROOT / 'models' / 'experiment_logs' / f'pipeline_min_{RUN_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=True)


def run_cmd(cmd, name, extra_env=None, check=True):
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    print('$ ' + ' '.join(cmd))
    res = subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env, text=True, capture_output=True)
    (RUN_DIR / f'{name}.stdout.log').write_text(res.stdout or '')
    (RUN_DIR / f'{name}.stderr.log').write_text(res.stderr or '')
    (RUN_DIR / f'{name}.meta.json').write_text(json.dumps({'cmd': cmd, 'returncode': res.returncode}, indent=2))
    if res.stdout:
        print(res.stdout[-4000:])
    if res.stderr:
        print('--- stderr ---')
        print(res.stderr[-4000:])
    if check and res.returncode != 0:
        raise RuntimeError(f'{name} failed with code {res.returncode}')
    return res


def snapshot_model(tag):
    src_model = PROJECT_ROOT / 'models' / 'cfn_emb.pth'
    src_scaler = PROJECT_ROOT / 'models' / 'cfn_scaler.pkl'
    if src_model.exists():
        shutil.copy2(src_model, RUN_DIR / f'{tag}.cfn_emb.pth')
    if src_scaler.exists():
        shutil.copy2(src_scaler, RUN_DIR / f'{tag}.cfn_scaler.pkl')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PY_BIN:', PY_BIN)
print('RUN_DIR:', RUN_DIR)


In [ ]:
# 1) Base retrain (all sources)
run_cmd([
    PY_BIN, '-m', 'src.training.train_cfn',
    '--data', 'data/processed/causal_multimodal_dataset.csv',
    '--train-source', 'all',
    '--use-embeddings', '--use-scaler',
    '--group-balance', '--use-weighted-sampler',
    '--loss', 'focal', '--focal-alpha', '0.75', '--focal-gamma', '2.0',
    '--causal-weight', '0.15',
    '--scheduler', 'cosine',
    '--epochs', '30', '--patience', '8', '--batch-size', '128',
    '--lr', '3e-4', '--weight-decay', '1e-4',
    '--selection-metric', 'hybrid_robust',
    '--selection-threshold', '0.5',
    '--min-domain-spec', '0.30',
    '--min-domain-rec', '0.50',
], '01_train_base')
snapshot_model('01_train_base')


In [ ]:
# 2) Mixed-domain constrained sweep for base model
cache_base = PROJECT_ROOT / 'eval_cache_mixed_base.json'
if cache_base.exists():
    cache_base.unlink()

run_cmd([
    PY_BIN, 'scripts/mixed_domain_sweep.py',
    '--build-manifest',
    '--manifest', 'eval_manifest_mixed.tsv',
    '--cache', 'eval_cache_mixed_base.json',
    '--out-json', 'models/mixed_sweep_base.json',
    '--fakeav-root', 'data/raw/fakeavceleb',
    '--dfdc-root', 'data/raw/dfdc/train_sample_videos',
    '--per-class-per-dataset', '200',
    '--min-rec', '0.75', '--min-spec', '0.50',
    '--selection-objective', 'hybrid_robust',
    '--min-domain-rec', '0.50',
    '--min-domain-spec', '0.30',
    '--allow-fallback',
], '02_sweep_base')


In [ ]:
# 3) Optional boost: build DFDC-weighted CSV
import pandas as pd

src_csv = PROJECT_ROOT / 'data' / 'processed' / 'causal_multimodal_dataset.csv'
out_csv = PROJECT_ROOT / 'data' / 'processed' / 'causal_multimodal_dataset_dfdc_boost.csv'
DFDC_FACTOR = 8

full = pd.read_csv(src_csv)
if 'dataset' in full.columns:
    dfdc = full[full['dataset'].astype(str).str.lower() == 'dfdc']
else:
    dfdc = full[full.get('path', '').astype(str).str.lower().str.contains('dfdc', regex=False)]

boosted = pd.concat([full] + [dfdc.copy() for _ in range(max(DFDC_FACTOR - 1, 0))], ignore_index=True)
boosted = boosted.sample(frac=1.0, random_state=42).reset_index(drop=True)
boosted.to_csv(out_csv, index=False)

print('Source rows:', len(full), '| DFDC rows:', len(dfdc), '| Boosted rows:', len(boosted))
print('Saved:', out_csv)


In [ ]:
# 4) Retrain boosted model
run_cmd([
    PY_BIN, '-m', 'src.training.train_cfn',
    '--data', 'data/processed/causal_multimodal_dataset_dfdc_boost.csv',
    '--train-source', 'all',
    '--use-embeddings', '--use-scaler',
    '--group-balance', '--use-weighted-sampler',
    '--loss', 'focal', '--focal-alpha', '0.75', '--focal-gamma', '2.0',
    '--causal-weight', '0.15',
    '--scheduler', 'cosine',
    '--epochs', '35', '--patience', '10', '--batch-size', '128',
    '--lr', '2e-4', '--weight-decay', '1e-4',
    '--selection-metric', 'hybrid_robust',
    '--selection-threshold', '0.5',
    '--min-domain-spec', '0.30',
    '--min-domain-rec', '0.50',
], '03_train_dfdc_boost')
snapshot_model('03_train_dfdc_boost')


In [ ]:
# 5) Mixed-domain constrained sweep for boosted model
cache_boost = PROJECT_ROOT / 'eval_cache_mixed_boost.json'
if cache_boost.exists():
    cache_boost.unlink()

run_cmd([
    PY_BIN, 'scripts/mixed_domain_sweep.py',
    '--manifest', 'eval_manifest_mixed.tsv',
    '--cache', 'eval_cache_mixed_boost.json',
    '--out-json', 'models/mixed_sweep_boost.json',
    '--per-class-per-dataset', '200',
    '--min-rec', '0.75', '--min-spec', '0.50',
    '--selection-objective', 'hybrid_robust',
    '--min-domain-rec', '0.50',
    '--min-domain-spec', '0.30',
    '--allow-fallback',
], '04_sweep_boost')


In [ ]:
# 6) Compare base vs boosted and select best env
candidates = {
    'base': PROJECT_ROOT / 'models' / 'mixed_sweep_base.json',
    'boosted': PROJECT_ROOT / 'models' / 'mixed_sweep_boost.json',
}
rows = []
for name, path in candidates.items():
    if not path.exists():
        continue
    payload = json.loads(path.read_text())
    b = payload['best']
    ov = b['metrics']['overall']
    dfdc = b['metrics']['per_dataset'].get('DFDC', {})
    rows.append({
        'name': name,
        'path': str(path),
        'overall_bal_acc': ov.get('bal_acc', 0.0),
        'overall_f1': ov.get('f1', 0.0),
        'dfdc_bal_acc': dfdc.get('bal_acc', 0.0),
        'recommend_env': payload['recommend_env'],
        'payload': payload,
    })

if not rows:
    raise RuntimeError('No candidate sweep outputs found.')

rows = sorted(rows, key=lambda x: (x['dfdc_bal_acc'], x['overall_bal_acc'], x['overall_f1']), reverse=True)
selected = rows[0]
SELECTED_PAYLOAD = selected['payload']

print('==== MODEL COMPARISON ====')
for r in rows:
    print(f"{r['name']}: overall_bal_acc={r['overall_bal_acc']:.3f} overall_f1={r['overall_f1']:.3f} dfdc_bal_acc={r['dfdc_bal_acc']:.3f}")

print('Selected:', selected['name'])
print('Result path:', selected['path'])


In [ ]:
# 7) Apply selected env + save final report
for k, v in SELECTED_PAYLOAD['recommend_env'].items():
    os.environ[k] = v

best = SELECTED_PAYLOAD['best']
overall = best['metrics']['overall']
per_ds = best['metrics']['per_dataset']

print('Best config:')
print('  PROB=', best['prob'], 'RATIO=', best['ratio'], 'CAUSAL=', best['causal'], 'REQUIRE_FLAG=', best['require_flag'])
print('Overall:')
print('  Acc={acc:.3f} BalAcc={bal_acc:.3f} F1={f1:.3f} Rec={rec:.3f} Spec={spec:.3f}'.format(**overall))
print('Per dataset:')
for ds, m in per_ds.items():
    print(f"  [{ds}] Acc={m['acc']:.3f} BalAcc={m['bal_acc']:.3f} F1={m['f1']:.3f} Rec={m['rec']:.3f} Spec={m['spec']:.3f}")

print('Applied env:')
for k in ['CFN_PROB_THRESH', 'CFN_RATIO_THRESH', 'CFN_CAUSAL_THRESH', 'CFN_REQUIRE_FLAG']:
    print(k, '=', os.environ.get(k))

report = {
    'run_id': RUN_ID,
    'run_dir': str(RUN_DIR),
    'selected_metrics': overall,
    'per_dataset': per_ds,
    'recommend_env': SELECTED_PAYLOAD['recommend_env'],
}
report_path = RUN_DIR / 'final_report.json'
report_path.write_text(json.dumps(report, indent=2))
print('Saved report:', report_path)


In [ ]:
# 8) Train/apply video-level calibrator
selected_cache = sel.get('cache') if isinstance(sel, dict) else None
if not selected_cache:
    raise RuntimeError('Selected sweep payload not found. Run previous cell first.')

run_cmd([
    PY_BIN, 'scripts/train_video_calibrator.py',
    '--cache', selected_cache,
    '--out', 'models/video_calibrator.pkl',
    '--min-rec', '0.70',
    '--min-spec', '0.45',
], '07_train_video_calibrator')

import joblib
cal_payload = joblib.load(PROJECT_ROOT / 'models' / 'video_calibrator.pkl')
cal_thr = float(cal_payload.get('threshold', 0.5))
os.environ['CFN_VIDEO_CALIBRATOR_PATH'] = str(PROJECT_ROOT / 'models' / 'video_calibrator.pkl')
os.environ['CFN_CALIBRATOR_THRESH'] = f'{cal_thr:.4f}'

print('CFN_VIDEO_CALIBRATOR_PATH=', os.environ['CFN_VIDEO_CALIBRATOR_PATH'])
print('CFN_CALIBRATOR_THRESH=', os.environ['CFN_CALIBRATOR_THRESH'])
